# COVID-19 Data Analysis Using Python

## Project Overview
This project analyzes cumulative COVID-19 confirmed cases, deaths, and recoveries across countries and regions.

The analysis covers:
- Data loading and exploration
- Data cleaning and missing-value handling
- Exploratory data analysis (EDA)
- Time-series analysis
- Data transformation from wide to long format
- Dataset merging
- Recovery and death-rate calculations
- Country and regional comparisons
- Data visualization

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn


## Dataset

The case-study dataset contains three Excel sheets:

- `covid_19_confirmed_v1` — cumulative confirmed cases
- `covid_19_deaths_v1` — cumulative deaths
- `covid_19_recovered_v1` — cumulative recovered cases

The provided case-study brief describes daily data from **January 22, 2020 to May 29, 2021** and more than 276 geographic entries.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")


## 1. Load the Data

In [ ]:
covid = pd.read_excel(
    "data/covid_19_dataset.xlsx",
    sheet_name=None
)

covid_19_deaths = covid["covid_19_deaths_v1"].copy()
covid_19_confirmed = covid["covid_19_confirmed_v1"].copy()
covid_19_recovered = covid["covid_19_recovered_v1"].copy()

print("Deaths:", covid_19_deaths.shape)
print("Confirmed:", covid_19_confirmed.shape)
print("Recovered:", covid_19_recovered.shape)


## 2. Standardize Date Columns

In [ ]:
dates = pd.date_range(
    start="2020-01-22",
    end="2021-05-29",
    freq="D"
)

fixed_columns = ["Province/State", "Country/Region", "Lat", "Long"]
new_columns = fixed_columns + list(dates)

for df in [covid_19_deaths, covid_19_confirmed, covid_19_recovered]:
    df.columns = new_columns

date_columns = list(dates)
latest_date = date_columns[-1]

print("Date columns:", len(date_columns))
print("Latest date:", latest_date)


## 3. Explore Dataset Structure

In [ ]:
for name, df in {
    "Deaths": covid_19_deaths,
    "Confirmed": covid_19_confirmed,
    "Recovered": covid_19_recovered
}.items():
    print(f"===== {name} =====")
    print("Shape:", df.shape)
    display(df.head())
    df.info()
    print()


## 4. Missing Values and Data Cleaning

In [ ]:
missing_before = {
    "Confirmed": covid_19_confirmed[date_columns].isna().sum().sum(),
    "Deaths": covid_19_deaths[date_columns].isna().sum().sum(),
    "Recovered": covid_19_recovered[date_columns].isna().sum().sum()
}

print("Missing values before imputation:", missing_before)

# Forward-fill time-series values across dates.
for df in [covid_19_confirmed, covid_19_deaths, covid_19_recovered]:
    df[date_columns] = df[date_columns].ffill(axis=1)

# Standardize blank province/state values.
for df in [covid_19_confirmed, covid_19_deaths, covid_19_recovered]:
    df["Province/State"] = df["Province/State"].fillna("All Provinces")

missing_after = {
    "Confirmed": covid_19_confirmed[date_columns].isna().sum().sum(),
    "Deaths": covid_19_deaths[date_columns].isna().sum().sum(),
    "Recovered": covid_19_recovered[date_columns].isna().sum().sum()
}

print("Missing values after imputation:", missing_after)


## 5. Confirmed Cases: Top Countries

In [ ]:
country_confirmed = (
    covid_19_confirmed
    .groupby("Country/Region")[date_columns]
    .sum()
)

latest_country_confirmed = (
    country_confirmed[latest_date]
    .sort_values(ascending=False)
)

top_5_names = latest_country_confirmed.head(5).index
top_5_data = country_confirmed.loc[top_5_names]

display(latest_country_confirmed.head(5))


In [ ]:
plt.figure(figsize=(14, 7))

for country in top_5_names:
    plt.plot(
        top_5_data.columns,
        top_5_data.loc[country],
        label=country
    )

plt.title("Cumulative Confirmed COVID-19 Cases - Top 5 Countries")
plt.xlabel("Date")
plt.ylabel("Confirmed Cases")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Confirmed Cases in China

In [ ]:
china_confirmed = country_confirmed.loc["China"]

plt.figure(figsize=(14, 6))
plt.plot(china_confirmed.index, china_confirmed.values, label="China")

plt.title("Cumulative Confirmed COVID-19 Cases in China")
plt.xlabel("Date")
plt.ylabel("Confirmed Cases")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Peak Daily New Cases: Germany, France, and Italy

In [ ]:
selected_countries = country_confirmed.loc[
    ["Germany", "France", "Italy"]
]

daily_new_cases = selected_countries.diff(axis=1)

peak_analysis = pd.DataFrame({
    "Peak Daily New Cases": daily_new_cases.max(axis=1),
    "Peak Date": daily_new_cases.idxmax(axis=1)
})

display(peak_analysis)


## 8. Recovery Rate: Canada vs Australia on December 31, 2020

In [ ]:
comparison_date = pd.Timestamp("2020-12-31")

results = []

for country in ["Canada", "Australia"]:
    confirmed = covid_19_confirmed.loc[
        covid_19_confirmed["Country/Region"] == country,
        comparison_date
    ].sum()

    recovered = covid_19_recovered.loc[
        covid_19_recovered["Country/Region"] == country,
        comparison_date
    ].sum()

    recovery_rate = recovered / confirmed * 100

    results.append({
        "Country": country,
        "Confirmed": confirmed,
        "Recovered": recovered,
        "Recovery Rate (%)": recovery_rate
    })

recovery_comparison = pd.DataFrame(results)
display(recovery_comparison)


## 9. Death Rate Distribution in Canadian Provinces

In [ ]:
canada_deaths = covid_19_deaths[
    covid_19_deaths["Country/Region"] == "Canada"
][["Province/State", latest_date]].rename(columns={latest_date: "Deaths"})

canada_confirmed = covid_19_confirmed[
    covid_19_confirmed["Country/Region"] == "Canada"
][["Province/State", latest_date]].rename(columns={latest_date: "Confirmed"})

canada_rates = canada_deaths.merge(
    canada_confirmed,
    on="Province/State",
    how="inner"
)

canada_rates = canada_rates[canada_rates["Confirmed"] > 0].copy()

canada_rates["Death Rate (%)"] = (
    canada_rates["Deaths"] / canada_rates["Confirmed"]
) * 100

display(
    canada_rates.sort_values(
        "Death Rate (%)",
        ascending=False
    )
)


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(y=canada_rates["Death Rate (%)"])

plt.title("Distribution of Death Rates Among Canadian Provinces")
plt.ylabel("Death Rate (%)")
plt.tight_layout()
plt.show()


## 10. Transform Death Data from Wide to Long Format

In [ ]:
death_long = covid_19_deaths.melt(
    id_vars=fixed_columns,
    var_name="Date",
    value_name="Deaths"
)

death_long["Date"] = pd.to_datetime(death_long["Date"])

death_country = (
    death_long
    .groupby(["Country/Region", "Date"], as_index=False)["Deaths"]
    .sum()
)

deaths_transformed = (
    death_country
    .pivot(
        index="Date",
        columns="Country/Region",
        values="Deaths"
    )
    .reset_index()
)

print("Date data type:", deaths_transformed["Date"].dtype)
display(deaths_transformed.tail())


## 11. Total Deaths by Country

In [ ]:
total_deaths_by_country = (
    covid_19_deaths
    .groupby("Country/Region")[latest_date]
    .sum()
    .sort_values(ascending=False)
)

display(total_deaths_by_country.head(10))


## 12. Average Daily Deaths

In [ ]:
country_deaths = (
    covid_19_deaths
    .groupby("Country/Region")[date_columns]
    .sum()
)

daily_deaths = country_deaths.diff(axis=1)

average_daily_deaths = (
    daily_deaths
    .mean(axis=1)
    .sort_values(ascending=False)
)

display(average_daily_deaths.head(5))


## 13. United States: Total Deaths Over Time

In [ ]:
us_deaths = covid_19_deaths[
    covid_19_deaths["Country/Region"] == "US"
]

us_total_deaths = us_deaths[date_columns].sum()

plt.figure(figsize=(12, 6))
plt.plot(
    us_total_deaths.index,
    us_total_deaths.values,
    label="US Total Deaths"
)

plt.title("Total COVID-19 Deaths in the United States Over Time")
plt.xlabel("Date")
plt.ylabel("Total Deaths")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 14. Merge Confirmed, Deaths, and Recovered Data

In [ ]:
def to_country_long(df, value_name):
    long_df = df.melt(
        id_vars=fixed_columns,
        var_name="Date",
        value_name=value_name
    )

    long_df["Date"] = pd.to_datetime(long_df["Date"])

    return (
        long_df
        .groupby(["Country/Region", "Date"], as_index=False)[value_name]
        .sum()
    )

confirmed_country = to_country_long(
    covid_19_confirmed, "Confirmed"
)

deaths_country = to_country_long(
    covid_19_deaths, "Deaths"
)

recovered_country = to_country_long(
    covid_19_recovered, "Recovered"
)

merged = (
    confirmed_country
    .merge(
        deaths_country,
        on=["Country/Region", "Date"],
        how="left"
    )
    .merge(
        recovered_country,
        on=["Country/Region", "Date"],
        how="left"
    )
    .sort_values(["Country/Region", "Date"])
)

display(merged.head())
print("Merged shape:", merged.shape)


## 15. Monthly New Cases, Deaths, and Recoveries

In [ ]:
for column in ["Confirmed", "Deaths", "Recovered"]:
    merged[f"New {column}"] = (
        merged.groupby("Country/Region")[column].diff()
    )

merged["Month"] = merged["Date"].dt.to_period("M")

monthly_data = (
    merged
    .groupby(["Country/Region", "Month"])[
        ["New Confirmed", "New Deaths", "New Recovered"]
    ]
    .sum()
    .reset_index()
)

display(monthly_data.head(10))


## 16. Monthly Analysis for US, Italy, and Brazil

In [ ]:
selected_monthly = monthly_data[
    monthly_data["Country/Region"].isin(
        ["US", "Italy", "Brazil"]
    )
]

display(selected_monthly)


## 17. Average Death Rate During 2020

In [ ]:
merged_2020 = merged[
    (merged["Date"].dt.year == 2020) &
    (merged["Confirmed"] > 0)
].copy()

merged_2020["Death Rate (%)"] = (
    merged_2020["Deaths"] /
    merged_2020["Confirmed"]
) * 100

average_death_rate_2020 = (
    merged_2020
    .groupby("Country/Region")["Death Rate (%)"]
    .mean()
    .sort_values(ascending=False)
)

display(average_death_rate_2020.head(10))


## 18. South Africa: Recoveries vs Deaths

In [ ]:
latest_merged_date = merged["Date"].max()

south_africa_latest = merged[
    (merged["Country/Region"] == "South Africa") &
    (merged["Date"] == latest_merged_date)
]

south_africa_summary = south_africa_latest[
    ["Recovered", "Deaths"]
].sum()

display(south_africa_summary.to_frame("Total"))


## 19. US Monthly Recovery Ratio

In [ ]:
us = merged[
    (merged["Country/Region"] == "US") &
    merged["Date"].between("2020-03-01", "2021-05-31")
].copy()

us["Month"] = us["Date"].dt.to_period("M")

# Use the latest cumulative observation in each month.
us_month_end = (
    us.sort_values("Date")
    .groupby("Month")
    .tail(1)
    .copy()
)

us_month_end["Recovery Ratio (%)"] = (
    us_month_end["Recovered"] /
    us_month_end["Confirmed"]
) * 100

display(
    us_month_end[
        ["Month", "Confirmed", "Recovered", "Recovery Ratio (%)"]
    ]
)


## 20. Key Skills Demonstrated

- Python for data analysis
- Pandas for data loading, cleaning, grouping, reshaping, merging, and aggregation
- NumPy for numerical operations
- Matplotlib and Seaborn for visualization
- Missing-value handling
- Time-series analysis
- Wide-to-long data transformation
- Country-level and regional analysis
- KPI and rate calculations
- Data integration and exploratory data analysis

## Notes

This project is a portfolio case study based on the supplied dataset and case-study questions. Some COVID-19 reporting series can contain revisions or reporting artifacts, so daily changes and rate calculations should be interpreted as analysis of the supplied dataset rather than as an authoritative epidemiological assessment.
